# Data Preprocessing and Baseline MLP — Steel Plates Faults

The goal of this notebook is to prepare the Steel Plates Faults dataset for later neural network classification.

The preprocessing focuses on:
- loading and cleaning the dataset
- separating features and target
- train/validation/test split
- checking split balance
- removal of constant and perfectly redundant features
- detection of binary and continuous variables
- transformation of strongly skewed variables
- normalization of continuous features
- encoding target labels
- final validation of processed data

The preprocessing decisions are based on the findings from the exploratory data analysis. Since the later model will be an MLP, particular attention is given to scaling and distribution shape of the input features.

In [37]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import arff

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

## Load the dataset

In [38]:
dataset_path = Path("../dataset/php9xWOpn.arff")

data, meta = arff.loadarff(dataset_path)
df = pd.DataFrame(data)

print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1941 entries, 0 to 1940
Data columns (total 34 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   V1      1941 non-null   float64
 1   V2      1941 non-null   float64
 2   V3      1941 non-null   float64
 3   V4      1941 non-null   float64
 4   V5      1941 non-null   float64
 5   V6      1941 non-null   float64
 6   V7      1941 non-null   float64
 7   V8      1941 non-null   float64
 8   V9      1941 non-null   float64
 9   V10     1941 non-null   float64
 10  V11     1941 non-null   float64
 11  V12     1941 non-null   float64
 12  V13     1941 non-null   float64
 13  V14     1941 non-null   float64
 14  V15     1941 non-null   float64
 15  V16     1941 non-null   float64
 16  V17     1941 non-null   float64
 17  V18     1941 non-null   float64
 18  V19     1941 non-null   float64
 19  V20     1941 non-null   float64
 20  V21     1941 non-null   float64
 21  V22     1941 non-null   float64
 22  V23     194

Convert `Class` column from object to string.

In [39]:
df["Class"] = df["Class"].apply(lambda x: x.decode("utf-8") if isinstance(x, bytes) else x).astype("string")
print(df["Class"].dtype)

string


## Description of the dataframe

In [40]:
print(df.head())
print(df.shape)
print(df.columns.tolist())
print(df["Class"].value_counts())

       V1      V2         V3         V4      V5    V6     V7        V8    V9  \
0    42.0    50.0   270900.0   270944.0   267.0  17.0   44.0   24220.0  76.0   
1   645.0   651.0  2538079.0  2538108.0   108.0  10.0   30.0   11397.0  84.0   
2   829.0   835.0  1553913.0  1553931.0    71.0   8.0   19.0    7972.0  99.0   
3   853.0   860.0   369370.0   369415.0   176.0  13.0   45.0   18996.0  99.0   
4  1289.0  1306.0   498078.0   498335.0  2409.0  60.0  260.0  246930.0  37.0   

     V10  ...     V25     V26     V27  V28  V29  V30  V31  V32  V33  Class  
0  108.0  ...  0.8182 -0.2913  0.5822  1.0  0.0  0.0  0.0  0.0  0.0      1  
1  123.0  ...  0.7931 -0.1756  0.2984  1.0  0.0  0.0  0.0  0.0  0.0      1  
2  125.0  ...  0.6667 -0.1228  0.2150  1.0  0.0  0.0  0.0  0.0  0.0      1  
3  126.0  ...  0.8444 -0.1568  0.5212  1.0  0.0  0.0  0.0  0.0  0.0      1  
4  126.0  ...  0.9338 -0.1992  1.0000  1.0  0.0  0.0  0.0  0.0  0.0      1  

[5 rows x 34 columns]
(1941, 34)
['V1', 'V2', 'V3', 'V4'

## Separate features and target

In [41]:
X = df.drop(columns=["Class"]).copy()
y = df["Class"].copy()

print(X.shape)
print(y.shape)

(1941, 33)
(1941,)


## Split the data into train, validation, and test sets

The dataset is split before any transformation so that all preprocessing steps can be fitted only on the training set and then applied to validation and test data. This prevents data leakage.

In [42]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Validation shape:", X_val.shape, y_val.shape)
print("Test shape:", X_test.shape, y_test.shape)

Train shape: (1358, 33) (1358,)
Validation shape: (291, 33) (291,)
Test shape: (292, 33) (292,)


## Check class distribution after split

In [43]:
print("Train class distribution")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

print("\nValidation class distribution")
print(y_val.value_counts())
print(y_val.value_counts(normalize=True))

print("\nTest class distribution")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True))

Train class distribution
Class
1    887
2    471
Name: count, dtype: Int64
Class
1    0.653166
2    0.346834
Name: proportion, dtype: Float64

Validation class distribution
Class
1    190
2    101
Name: count, dtype: Int64
Class
1    0.652921
2    0.347079
Name: proportion, dtype: Float64

Test class distribution
Class
1    191
2    101
Name: count, dtype: Int64
Class
1    0.65411
2    0.34589
Name: proportion, dtype: Float64


### Conclusion

The class distributions in the training, validation, and test sets remain very similar due to stratified splitting. This ensures that all subsets preserve the original class proportions and can be compared more fairly during later modeling and evaluation.

## Check missing values in each split

In [44]:
print("Missing values in training set")
print(X_train.isnull().sum().sort_values(ascending=False).head(10))

print("\nMissing values in validation set")
print(X_val.isnull().sum().sort_values(ascending=False).head(10))

print("\nMissing values in test set")
print(X_test.isnull().sum().sort_values(ascending=False).head(10))

Missing values in training set
V1     0
V18    0
V32    0
V31    0
V30    0
V29    0
V28    0
V27    0
V26    0
V25    0
dtype: int64

Missing values in validation set
V1     0
V18    0
V32    0
V31    0
V30    0
V29    0
V28    0
V27    0
V26    0
V25    0
dtype: int64

Missing values in test set
V1     0
V18    0
V32    0
V31    0
V30    0
V29    0
V28    0
V27    0
V26    0
V25    0
dtype: int64


### Conclusion

The split datasets do not show any missing-value issues. Therefore, no imputation step is required in the preprocessing pipeline.

## Remove constant features

Features with only one unique value in the training set do not provide any useful information for classification and can be removed.

In [45]:
constant_cols = [col for col in X_train.columns if X_train[col].nunique(dropna=True) <= 1]

X_train = X_train.drop(columns=constant_cols)
X_val = X_val.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols)

print("Removed constant columns:", constant_cols)
print("Remaining number of features:", X_train.shape[1])

Removed constant columns: []
Remaining number of features: 33


### Conclusion

Constant features, if any are present, are removed because they do not contribute to class separation and only add unnecessary dimensionality.

## Remove perfectly redundant features

The EDA showed that some variables are perfectly or almost perfectly redundant. For the initial preprocessing, only perfectly correlated features are removed.

In [46]:
corr = X_train.corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

perfect_corr_cols = [col for col in upper.columns if any(np.isclose(upper[col], 1.0))]

X_train = X_train.drop(columns=perfect_corr_cols)
X_val = X_val.drop(columns=perfect_corr_cols)
X_test = X_test.drop(columns=perfect_corr_cols)

print("Removed perfectly correlated columns:", perfect_corr_cols)
print("Remaining number of features:", X_train.shape[1])

Removed perfectly correlated columns: ['V4', 'V13']
Remaining number of features: 31


### Conclusion

Perfectly redundant variables are removed to reduce duplicated information in the feature space. This simplifies the input representation without losing meaningful predictive content.

## Separate binary and continuous features

The EDA indicated that several variables behave like binary or sparse indicator variables, while others are continuous. These groups should be treated differently during preprocessing.

In [47]:
binary_cols = [
    col for col in X_train.columns
    if set(pd.Series(X_train[col]).dropna().unique()).issubset({0, 1})
]

continuous_cols = [col for col in X_train.columns if col not in binary_cols]

print("Binary columns:", binary_cols)
print("Number of binary columns:", len(binary_cols))
print("Continuous columns:", continuous_cols)
print("Number of continuous columns:", len(continuous_cols))

Binary columns: ['V12', 'V28', 'V29', 'V30', 'V31', 'V32', 'V33']
Number of binary columns: 7
Continuous columns: ['V1', 'V2', 'V3', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27']
Number of continuous columns: 24


### Conclusion

The features can be divided into binary indicator variables and continuous numerical variables. This distinction is important because binary variables should generally remain unchanged, while continuous variables may require transformation and scaling.

## Detect strongly right-skewed continuous variables

The EDA showed that several continuous variables are strongly right-skewed. To reduce skewness and the influence of extreme large values, a `log1p` transformation is applied to strongly skewed non-negative continuous features.

In [48]:
skewness = X_train[continuous_cols].skew()

skewed_cols = [
    col for col in continuous_cols
    if (X_train[col] >= 0).all() and skewness[col] > 1
]

print("Skewed continuous columns selected for log1p transformation:", skewed_cols)

Skewed continuous columns selected for log1p transformation: ['V3', 'V5', 'V6', 'V7', 'V8', 'V10', 'V14', 'V18']


### Conclusion

Only strongly right-skewed continuous variables with non-negative values are selected for log transformation. This keeps the preprocessing targeted and avoids applying unnecessary transformations to all variables.

## Apply log transformation to selected columns

In [49]:
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()

for col in skewed_cols:
    X_train[col] = np.log1p(X_train[col])
    X_val[col] = np.log1p(X_val[col])
    X_test[col] = np.log1p(X_test[col])

print("Log-transformed columns:", skewed_cols)

Log-transformed columns: ['V3', 'V5', 'V6', 'V7', 'V8', 'V10', 'V14', 'V18']


### Conclusion

The `log1p` transformation compresses extreme large values and makes the selected feature distributions less skewed, which is beneficial for later neural network training.

## Normalize continuous features

Since MLPs are sensitive to feature scale, continuous variables are standardized using `StandardScaler`. The scaler is fitted only on the training set and then applied to validation and test sets.

In [50]:
scaler = StandardScaler()

X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_val[continuous_cols] = scaler.transform(X_val[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print(X_train[continuous_cols].describe().T[["mean", "std"]].head())

            mean       std
V1  8.894865e-17  1.000368
V2 -1.569682e-17  1.000368
V3 -1.778319e-15  1.000368
V5  2.092909e-16  1.000368
V6 -1.935941e-16  1.000368


### Conclusion

Continuous variables are standardized so that they have comparable scale. This improves optimization stability and helps prevent variables with large ranges from dominating the learning process.

## Convert feature matrices to float32

In [51]:
X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
X_test = X_test.astype(np.float32)

print(X_train.dtypes.value_counts())

float32    31
Name: count, dtype: int64


### Conclusion

The processed feature matrices are converted to `float32`, which is the standard numerical type used in most neural network frameworks.

## Encode target labels

The target labels are converted from strings to numeric values so that they can be used in classification models.

In [52]:
label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

print("Class mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))
print("Encoded training labels shape:", y_train_enc.shape)

Class mapping: {'1': np.int64(0), '2': np.int64(1)}
Encoded training labels shape: (1358,)


### Conclusion

The class labels are successfully encoded into numeric form. This makes the target variable compatible with later model training and evaluation.

## Final validation of processed data

In [53]:
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

print("y_train shape:", y_train_enc.shape)
print("y_val shape:", y_val_enc.shape)
print("y_test shape:", y_test_enc.shape)

print("NaNs in X_train:", np.isnan(X_train.to_numpy()).sum())
print("NaNs in X_val:", np.isnan(X_val.to_numpy()).sum())
print("NaNs in X_test:", np.isnan(X_test.to_numpy()).sum())

print("NaNs in y_train:", np.isnan(y_train_enc).sum())
print("NaNs in y_val:", np.isnan(y_val_enc).sum())
print("NaNs in y_test:", np.isnan(y_test_enc).sum())

X_train shape: (1358, 31)
X_val shape: (291, 31)
X_test shape: (292, 31)
y_train shape: (1358,)
y_val shape: (291,)
y_test shape: (292,)
NaNs in X_train: 0
NaNs in X_val: 0
NaNs in X_test: 0
NaNs in y_train: 0
NaNs in y_val: 0
NaNs in y_test: 0


## Final preprocessing conclusion

The preprocessing pipeline prepared the Steel Plates Faults dataset for later neural network classification. The data was first split into training, validation, and test sets using stratified sampling to preserve the original class proportions and to avoid data leakage.

No major missing-value issues were found, so no imputation was needed. Constant features were removed because they do not contribute useful predictive information. Perfectly redundant variables were also removed to reduce duplicated information in the feature space.

The remaining variables were divided into binary and continuous groups. Strongly right-skewed non-negative continuous features were transformed using `log1p` in order to reduce skewness and the impact of extreme values. Continuous variables were then standardized using `StandardScaler`, while binary variables were left unchanged.

Finally, the feature matrices were converted to `float32` format and the target labels were encoded into numeric form. The resulting processed datasets are now ready for baseline MLP training and later experiments.